# Exporting Modelica Models to FMUs

## Overview

This notebook demonstrates how to export Modelica models to FMUs using the `OMPython` library. The process involves identifying the Modelica models, compiling them into FMUs, and organizing the output files.

In [1]:
import sys
from typing import Literal
from pathlib import Path
from shutil import move

from OMPython import ModelicaSystem

REPO_ROOT = Path().cwd()
if REPO_ROOT.name != "SystemSimulation":
    raise RuntimeError("This script must be run from the root of the repository.")

PLATFORM = sys.platform
if PLATFORM.startswith("linux"):
    FMU_EXPORTER_PATH = REPO_ROOT / "demos/ControlledPendulum/artifacts/fmus/linux/"
elif PLATFORM.startswith("win"):
    FMU_EXPORTER_PATH = REPO_ROOT / "demos/ControlledPendulum/artifacts/fmus/windows/"
elif PLATFORM.startswith("darwin"):
    FMU_EXPORTER_PATH = REPO_ROOT / "demos/ControlledPendulum/artifacts/fmus/macos/"
else:
    raise RuntimeError(f"Unsupported platform: {PLATFORM}")
FMU_EXPORTER_PATH.mkdir(parents=True, exist_ok=True)

In [2]:
src_dir = Path.cwd() / "demos/ControlledPendulum/src/modelica/ControlledPendulum/"
main_pkg_name = src_dir.name
main_pkg_path = src_dir /'package.mo'

In [3]:
sub_pkgs = {}
for item in src_dir.iterdir():
    if item.is_dir():
        sub_pkgs[item.name] = {}
        for sub_sub_pkg in item.iterdir():
            if sub_sub_pkg.is_dir():
                sub_pkgs[item.name][sub_sub_pkg.name] = sub_sub_pkg
        if not sub_pkgs[item.name]:
            sub_pkgs[item.name] = item

sub_pkgs

{'Controllers': PosixPath('/home/flo/code/SystemSimulation/demos/ControlledPendulum/src/modelica/ControlledPendulum/Controllers'),
 'Examples': {'Contact': PosixPath('/home/flo/code/SystemSimulation/demos/ControlledPendulum/src/modelica/ControlledPendulum/Examples/Contact'),
  'NoContact': PosixPath('/home/flo/code/SystemSimulation/demos/ControlledPendulum/src/modelica/ControlledPendulum/Examples/NoContact')},
 'Actuators': PosixPath('/home/flo/code/SystemSimulation/demos/ControlledPendulum/src/modelica/ControlledPendulum/Actuators'),
 'Plants': PosixPath('/home/flo/code/SystemSimulation/demos/ControlledPendulum/src/modelica/ControlledPendulum/Plants'),
 'Sensors': PosixPath('/home/flo/code/SystemSimulation/demos/ControlledPendulum/src/modelica/ControlledPendulum/Sensors'),
 'Trajectories': PosixPath('/home/flo/code/SystemSimulation/demos/ControlledPendulum/src/modelica/ControlledPendulum/Trajectories')}

In [4]:
model_names = {}
for pkg_name, pkg in sub_pkgs.items():
    if isinstance(pkg, dict):
        model_names[pkg_name] = {}
        for sub_name, sub_pkg in pkg.items():
            model_names[pkg_name][sub_name] = {}
            for item in sub_pkg.iterdir():
                if item.is_file() and item.suffix == ".mo" and item.stem != "package":
                    model_name = f"{main_pkg_name}.{pkg_name}.{sub_name}.{item.stem}"
                    model_names[pkg_name][sub_name][item.stem] = model_name
    else:
        model_names[pkg_name] = {}
        for item in pkg.iterdir():
            if item.is_file() and item.suffix == ".mo" and item.stem != "package":
                model_name = f"{main_pkg_name}.{pkg_name}.{item.stem}"
                model_names[pkg_name][item.stem] = model_name

In [5]:
SOLVER = Literal["cvode", "euler"]
def create_fmu(package_file_path: Path,
               composed_model_name: str,
               solver: SOLVER,
               export_path: Path):
    """Create a ModelicaSystem instance for a given package and model.
    
    Args:
        package_file_path (Path): Path to the Modelica main package file (package.mo).
        composed_model_name (str): Name of the model to be instantiated (e.g., "MainPackageName.SubPackageName.ModelName").
        solver (LiteralString): The solver to be used for simulation (e.g., "cvode", "euler").
    Returns:
        ModelicaSystem: An instance of ModelicaSystem for the specified model.
    """
    modelica_system = ModelicaSystem(
        fileName=str(package_file_path),
        modelName=composed_model_name,
        commandLineOptions=f"--fmiFlags=s:{solver}",
    )

    modelica_system.buildModel()

    fmu_path = modelica_system.convertMo2Fmu(version="2.0", fmuType="cs")
    try:
        move(fmu_path, export_path)
        print(f"FMU created at: {export_path}")
    except Exception as e:
        print(f"Error moving FMU: {e}")

In [6]:
# Define Models that shall also be compiled with Euler solver
euler_models = ["PIDControllerReset", "Pendulum"]

In [16]:
for pkg_name, dir in model_names.items():
    export_dir = FMU_EXPORTER_PATH / pkg_name
    export_dir.mkdir(parents=True, exist_ok=True)

    for sub_pkg_name, sub_item in dir.items():
        if isinstance(sub_item, dict):
            for model_name, composed_model_name in sub_item.items():
                print(100 * '=')
                print(f"Creating FMU for model: {composed_model_name}")
                if model_name in euler_models:
                    create_fmu(main_pkg_path, composed_model_name, "euler", export_dir / f"{model_name}_euler.fmu")
                    create_fmu(main_pkg_path, composed_model_name, "cvode", export_dir / f"{model_name}_cvode.fmu")
                else:
                    solver = "cvode"
                    create_fmu(main_pkg_path, composed_model_name, solver, export_dir / f"{model_name}.fmu")
        else:
            model_name = sub_pkg_name
            composed_model_name = sub_item
            print(100 * '=')
            print(f"Creating FMU for model: {composed_model_name}")
            if model_name in euler_models:
                create_fmu(main_pkg_path, composed_model_name, "euler", export_dir / f"{model_name}_euler.fmu")
                create_fmu(main_pkg_path, composed_model_name, "cvode", export_dir / f"{model_name}_cvode.fmu")
            else:
                solver = "cvode"
                create_fmu(main_pkg_path, composed_model_name, solver, export_dir / f"{model_name}.fmu")

Creating FMU for model: ControlledPendulum.Controllers.PID_Continuous_old

Notification: Automatically loaded package Complex 4.0.0 due to uses annotation from Modelica.
Notification: Automatically loaded package ModelicaServices 4.0.0 due to uses annotation from Modelica.
Notification: Automatically loaded package Modelica 4.0.0 due to usage.


FMU created at: /home/flo/code/SystemSimulation/demos/ControlledPendulum/artifacts/fmus/linux/Controllers/PID_Continuous_old.fmu
Creating FMU for model: ControlledPendulum.Controllers.PIDController

Notification: Automatically loaded package Complex 4.0.0 due to uses annotation from Modelica.
Notification: Automatically loaded package ModelicaServices 4.0.0 due to uses annotation from Modelica.
Notification: Automatically loaded package Modelica 4.0.0 due to usage.


FMU created at: /home/flo/code/SystemSimulation/demos/ControlledPendulum/artifacts/fmus/linux/Controllers/PIDController.fmu
Creating FMU for model: ControlledPendulum.Controllers.P

In [7]:
export_dir = FMU_EXPORTER_PATH / 'Sensors'
export_dir.mkdir(parents=True, exist_ok=True)
model_name = model_names['Sensors']['AngleSensor']
create_fmu(main_pkg_path, model_name, "cvode", export_dir / f"AngleSensor.fmu")
model_name = model_names['Sensors']['AngleDecoder']
create_fmu(main_pkg_path, model_name, "cvode", export_dir / f"AngleDecoder.fmu")


Notification: Automatically loaded package Complex 4.0.0 due to uses annotation from Modelica.
Notification: Automatically loaded package ModelicaServices 4.0.0 due to uses annotation from Modelica.
Notification: Automatically loaded package Modelica 4.0.0 due to usage.


FMU created at: /home/flo/code/SystemSimulation/demos/ControlledPendulum/artifacts/fmus/linux/Sensors/AngleSensor.fmu

Notification: Automatically loaded package Complex 4.0.0 due to uses annotation from Modelica.
Notification: Automatically loaded package ModelicaServices 4.0.0 due to uses annotation from Modelica.
Notification: Automatically loaded package Modelica 4.0.0 due to usage.


FMU created at: /home/flo/code/SystemSimulation/demos/ControlledPendulum/artifacts/fmus/linux/Sensors/AngleDecoder.fmu


In [ ]:
export_dir = FMU_EXPORTER_PATH / 'Controllers'
export_dir.mkdir(parents=True, exist_ok=True)
model_name = model_names['Controllers']['PIDControllerReset']
create_fmu(main_pkg_path, model_name, "euler", export_dir / f"PIDControllerReset_euler.fmu")
create_fmu(main_pkg_path, model_name, "cvode", export_dir / f"PIDControllerReset_cvode.fmu")


Notification: Automatically loaded package Complex 4.0.0 due to uses annotation from Modelica.
Notification: Automatically loaded package ModelicaServices 4.0.0 due to uses annotation from Modelica.
Notification: Automatically loaded package Modelica 4.0.0 due to usage.


FMU created at: /home/flo/code/SystemSimulation/demos/ControlledPendulum/artifacts/fmus/linux/Controllers/PIDControllerReset_euler.fmu

Notification: Automatically loaded package Complex 4.0.0 due to uses annotation from Modelica.
Notification: Automatically loaded package ModelicaServices 4.0.0 due to uses annotation from Modelica.
Notification: Automatically loaded package Modelica 4.0.0 due to usage.


FMU created at: /home/flo/code/SystemSimulation/demos/ControlledPendulum/artifacts/fmus/linux/Controllers/PIDControllerReset_cvode.fmu
